## Clasificación de textos en español con LSTM y WikiCAT_esv2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leoe21/machine_learning_fundamentals/blob/main/03_unidad/RNN_Text/Sesion2/2-nlp-with-lstm-Foxtroot-team.ipynb)

En este notebook implementaremos un clasificador de textos en español utilizando la arquitectura de red LSTM, tomando como base el dataset **WikiCAT_esv2**. La idea es tener un punto de referencia clásico con LSTM para luego compararlo con modelos basados en transformers, usando la misma tarea de clasificación temática.

WikiCAT_esv2 es un corpus creado automáticamente a partir de artículos de Wikipedia y Wikidata. Cada ejemplo contiene el resumen de un artículo en español (`text`) y una etiqueta temática (`label`) entre 12 posibles categorías (Religión, Economía, Historia, etc.).

**Nota (aprendizaje):** Este cuaderno parte del **material original de la Sesión 2**. En cada sección se indica qué corresponde al **notebook original** y qué hemos **añadido o mejorado** en esta versión (embeddings con Spacy, pesos por clase, label smoothing, etc.).

#### Referencias
- Dataset: https://huggingface.co/datasets/PlanTL-GOB-ES/WikiCAT_esv2
- [Long Short-Term Memory](https://www.researchgate.net/publication/13853244_Long_Short-Term_Memory#fullTextFileContent)

In [1]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [2]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/leoe21/machine_learning_fundamentals/raw/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets

'test' is not recognized as an internal or external command,
operable program or batch file.


'test' is not recognized as an internal or external command,
operable program or batch file.
'test' is not recognized as an internal or external command,
operable program or batch file.
'test' is not recognized as an internal or external command,
operable program or batch file.
'test' is not recognized as an internal or external command,
operable program or batch file.


### Cargando el dataset WikiCAT_esv2  
**Original del notebook.**

En esta sección cargamos el dataset **WikiCAT_esv2** desde el HuggingFace Hub. Se trata de resúmenes de artículos de Wikipedia en español, cada uno etiquetado con una de 12 categorías temáticas (Religión, Economía, Historia, etc.).

La librería `datasets` de HuggingFace nos permite descargar y manejar este conjunto de datos de forma sencilla, devolviéndonos un objeto similar a un dataframe con los campos `text` (texto) y `label` (id numérico de la categoría).

In [3]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('PlanTL-GOB-ES/WikiCAT_esv2', split='train')
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 6716
})

Observemos uno de sus registros

In [4]:
dataset
dataset[0]

{'text': 'En estadística, un modelo probit es un tipo de regresión donde la variable dependiente puede tomar solo dos valores, por ejemplo, casados o no casados. La palabra es un acrónimo, viene de probabilidad + unit (unidad).[1]\u200b El propósito del modelo es estimar la probabilidad de que una observación con características particulares caerá en una categoría específica; además, clasificando las observaciones basadas en sus probabilidades predichas es un tipo de modelo de clasificación binario .\nUn modelo probit es una especificación popular para un modelo de respuesta ordinal[2]\u200b o binario. Como tal, trata el mismo conjunto de problemas que la regresión logística utilizando técnicas similares. El modelo probit, que emplea una función de enlace probit, se suele estimar utilizando el procedimiento estándar de máxima verosimilitud , que se denomina una regresión probit.\nLos modelos Probit fueron presentados por Chester Bliss en 1934;[3]\u200b  Ronald Fisher propuso un método 

In [5]:
dataset.features["label"].names

['Religión',
 'Entretenimiento',
 'Música',
 'Ciencia_y_Tecnología',
 'Política',
 'Economía',
 'Matemáticas',
 'Humanidades',
 'Deporte',
 'Derecho',
 'Historia',
 'Filosofía']

Para los efectos de esta tarea, nos servirán el texto y la categoría naturalmente.

A manera general, observemos que tan largos o cortos tienden a ser los textos.

In [6]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 54
Texto más largo: 14966
Longitud promedio: 1035.9082787373436


Estos valores son la cantidad de *caracteres* que tiene cada resumen de Wikipedia. Vemos que hay textos muy cortos (54 caracteres), textos bastante largos (casi 15k caracteres) y, en promedio, alrededor de 1k caracteres por ejemplo.

En el modelo usamos **tokens** (palabras) con longitud máxima **512**: truncar o rellenar con padding.

## Definiendo el Tokenizer  
**Original del notebook.**

- **Vocabulario por frecuencia**: las **50.000 palabras de mayor concurrencia** (`most_common`), para que los IDs privilegien las formas más vistas.
- **Sin quitar stopwords**: en LSTM, palabras como "el", "la", "de" aportan contexto; quitarlas aquí suele bajar la accuracy.
- Tokens especiales: `[PAD]` y `[UNK]`.

In [7]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúüñ]+", " ", text)
    return text.strip().split()

# Vocabulario: las 50k palabras de MAYOR frecuencia (most_common), no en orden arbitrario.
VOCAB_SIZE = 50_000
token_counts = Counter()
for text in dataset["text"]:
    token_counts.update(simple_tokenizer(text))

top_n_tokens = [token for token, _ in token_counts.most_common(VOCAB_SIZE - 2)]
vocab = {"[PAD]": 0, "[UNK]": 1}
for token in top_n_tokens:
    vocab[token] = len(vocab)

def tokenize_text(text, max_length=720):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok, vocab["[UNK]"]) for tok in tokens[:max_length]]
    ids += [vocab["[PAD]"]] * (max_length - len(ids))
    return ids

### Vectores preentrenados con Spacy
**Añadido en esta versión.**

Inicializamos la capa de embeddings del LSTM con vectores de palabras en español del modelo **es_core_news_md** de Spacy. Así el modelo parte de representaciones semánticas ya aprendidas en lugar de pesos aleatorios, lo que suele mejorar la accuracy en clasificación. **Lo realizamos de esta forma porque al querer replciar como este notebook original el accuracy no subia de 0,62**

In [8]:
# Instalar Spacy y modelo español con vectores (descomentar en Colab si hace falta)
# !pip install spacy
# !python -m spacy download es_core_news_lg

import spacy
import numpy as np
import torch

# es_core_news_lg: más vocabulario/vectores que _md => mejor cobertura para palabras raras
SPACY_MODEL = "es_core_news_lg"
try:
    nlp = spacy.load(SPACY_MODEL)
except OSError:
    import subprocess
    subprocess.run(["pip", "install", "spacy"], check=True)
    subprocess.run(["python", "-m", "spacy", "download", SPACY_MODEL], check=True)
    nlp = spacy.load(SPACY_MODEL)

# Dimensión de los vectores de Spacy (es_core_news_md/lg = 300)
EMB_DIM_SPACY = nlp.vocab.vectors_length
assert EMB_DIM_SPACY > 0, "El modelo debe incluir vectores (usa es_core_news_md o es_core_news_lg)."

# Matriz de embeddings: una fila por cada token del vocabulario
vocab_size = len(vocab)
embedding_matrix = np.zeros((vocab_size, EMB_DIM_SPACY), dtype=np.float32)
for token, idx in vocab.items():
    if token in ("[PAD]", "[UNK]"):
        continue
    # Obtener vector de la palabra (si no está, Spacy devuelve zeros)
    vec = nlp.vocab[token].vector
    embedding_matrix[idx] = vec

# [UNK]: inicializar con la media de los vectores del vocabulario (excl. PAD)
mask = np.ones(vocab_size, dtype=bool)
mask[vocab["[PAD]"]] = False
embedding_matrix[vocab["[UNK]"]] = embedding_matrix[mask].mean(axis=0)

spacy_embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)
print(f"Matriz de embeddings Spacy: {spacy_embedding_matrix.shape} (vocab_size x {EMB_DIM_SPACY})")

Matriz de embeddings Spacy: torch.Size([50000, 300]) (vocab_size x 300)


Exploremos ahora el tokenizador obtenido. *(Original.)*

In [9]:
print(f"Vocabulario: {len(vocab)} tokens")
print("Primeros 15 tokens:")
print(f"{top_n_tokens[:15]}")
print("15 tokens de en medio:")
print(f"{top_n_tokens[1000:1015]}")
print("Últimos 15 tokens:")
print(f"{top_n_tokens[-15:]}")

Vocabulario: 50000 tokens
Primeros 15 tokens:
['de', 'la', 'en', 'el', 'y', 'que', 'a', 'los', 'un', 'se', 'del', 'es', 'las', 'una', 'por']
15 tokens de en medio:
['religioso', 'protección', 'mitad', 'variedad', 'tecnológico', 'conocer', 'independencia', 'italia', 'soluciones', 'mejorar', 'regla', 'comprensión', 'fecha', 'usan', 'conocidos']
Últimos 15 tokens:
['balazar', 'basílica', 'profetizado', 'convocados', 'implorar', 'tropos', 'simbolismos', 'lint', 'terri', 'coeditora', 'datlow', 'mythic', 'endicott', 'tricksters', 'mitopoeias']


Los primeros tokens son los **más frecuentes** del corpus (`most_common`). Ejemplo mínimo de tokenización: *(Original.)*

In [10]:
tokenized = tokenize_text("hola mundo", max_length=6)
tokenized

[37024, 80, 0, 0, 0, 0]

Lo que obtenemos de vuelta son los ids de cada token según el vocabulario. Ahora algo importante que notamos aquí es el *padding*, durante el entrenamiento, queremos que las secuencias sean de tamaño fijo, para asi operar comodamente con matrices. Pero ya vimos que no todos los textos tienen la misma longitud. Entonces que hacer? para los que son más largos que una longitud dada simplemente cortamos, pero para los que son más cortos, debemos *rellenar* lo faltante con un *token especial de relleno o padding*. Y es justo lo que definimos allí, cuando la cadena es inferior a 8 **tokens**, entonces debemos hacer padding hasta que se cumplan los 8.

Si queremos saber a que token exactamente hacen referencia estos ids, simplemente revisamos el vocabulario que hemos construido: *(Original.)*

In [11]:
id_2_token = {v: k for k, v in vocab.items()}
[id_2_token[token] for token in tokenized]

['hola', 'mundo', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Claramente vemos los 3 tokens como cadenas independientes (el padding se considera un token independiente).

### Definiendo el dataset de pytorch  
**Original del notebook.**  
Ahora podemos proceder a definir el dataset. Esto debería ser muy sencillo dado que nuestro dataset es pequeño y ya tenemos el tokenizador listo.

In [12]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset

class WikiCATDataset(Dataset):

    def __init__(self, tokenizer, dataset, seq_length: int = 512):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.seq_length = seq_length
        # El propio dataset ya trae las etiquetas como enteros (0-11).
        # Aprovechamos la metainformación para conocer los nombres de clase.
        self.id2label = dataset.features["label"].names
        self.num_classes = len(self.id2label)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text = self.dataset[index]['text']
        y = self.dataset[index]['label']  # entero en [0, num_classes)
        data = {'input_ids': torch.tensor(self.tokenizer(text, max_length=self.seq_length))}
        data['y'] = torch.tensor(y)
        return data

    def __len__(self):
        return len(self.dataset)

Instanciamos el dataset con longitud máxima **720 tokens** (truncar/padding) para más contexto. *(Ajustado para mejorar accuracy.)*

In [13]:
max_len = 720
wikicat_dataset = WikiCATDataset(tokenize_text, dataset, seq_length=max_len)
assert len(wikicat_dataset) == len(dataset)

Y luego, procedemos a hacer el train-val-test split y crear los dataloaders. *(Original; en esta versión batch_size=32.)*

**Añadido en esta versión:** Pesos por clase para compensar el desbalance entre las 12 categorías (las menos frecuentes reciben mayor peso en la pérdida).

In [14]:
import random
import numpy as np
import torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader

# Reproducibilidad: misma semilla => mismo split y mismo orden de batches (evita variar 67% vs 71%)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Subimos el batch_size de 4 a 32 para mayor estabilidad en el entrenamiento
batch_size = 32

generator_split = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(wikicat_dataset, lengths=[0.8, 0.1, 0.1], generator=generator_split)
generator_train = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, generator=generator_train)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [15]:
import numpy as np
import torch

# Extraemos las etiquetas del train_dataset
train_labels = [dataset[i]['label'] for i in train_dataset.indices]

# Calculamos la frecuencia de cada clase
class_counts = np.bincount(train_labels)
total_samples = len(train_labels)
num_classes = len(class_counts)

# Calculamos los pesos: clases menos frecuentes tendrán mayor peso
class_weights = total_samples / (num_classes * class_counts)
class_weights_tensor = torch.FloatTensor(class_weights).to("cuda" if torch.cuda.is_available() else "cpu")

print("Pesos por clase:", class_weights_tensor)

Pesos por clase: tensor([1.8502, 3.3414, 1.0365, 0.5661, 0.4527, 0.7814, 0.9328, 1.1166, 1.7288,
        1.0246, 1.4036, 1.4169], device='cuda:0')


## Definición del modelo LSTM  
**Original:** arquitectura (LSTM bidireccional, clasificador denso). **Añadido:** inicialización de embeddings con vectores Spacy.

El modelo consta de: **Embedding** Spacy (es_core_news_lg, 300 dim) → **LSTM bidireccional** (3 capas, hidden 256) → **last_hidden + mean pooling** (usa toda la secuencia, sin promediar padding) → **Clasificador** (1024→512→512→256→12). LR 5e-4, hasta 40 épocas, patience 7.

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMBlock(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2, bidirectional=True, embedding_weights=None):
        super().__init__()
        if embedding_weights is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_weights, padding_idx=0, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout, bidirectional=bidirectional)
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)  # output: (batch, seq_len, hidden*2)
        # Último estado (como antes)
        if self.bidirectional:
            last_h = torch.cat((hidden[-2], hidden[-1]), dim=-1)  # (batch, hidden_dim*2)
        else:
            last_h = hidden[-1]
        # Mean pooling sobre toda la secuencia (ignorando padding) => usa todo el texto
        mask = (x != 0).float().unsqueeze(-1)  # (batch, seq_len, 1)
        mean_pool = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return torch.cat([last_h, mean_pool], dim=-1)  # (batch, hidden_dim*4) si bidir


### Definición del clasificador (LightningModule)  
**Original:** estructura del módulo. **Añadido:** `class_weights`, label smoothing 0.1, ReduceLROnPlateau, early stopping.

Modelo completo: tokenización → LSTM bidireccional → clasificador denso. AdamW, early stopping en `val-loss` (paciencia 5), hasta 25 épocas.

In [17]:
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy
import torch.nn.functional as F
import torch.nn as nn
import torch

class SpanishNewsClassifierWithLSTM(LightningModule):

    def __init__(self, vocab_size: int, num_classes: int, emb_dim: int, hidden_dim: int = 256, num_layers: int = 3, dropout: float = 0.25, bidirectional: bool = True, class_weights=None, embedding_weights=None):
        super(SpanishNewsClassifierWithLSTM, self).__init__()
        self.num_classes = num_classes
        self.class_weights = class_weights
        self.lstm = LSTMBlock(vocab_size, emb_dim, hidden_dim, num_classes, num_layers=num_layers, dropout=dropout, bidirectional=bidirectional, embedding_weights=embedding_weights)
        # last_hidden + mean_pool => el doble de dimensión
        lstm_out_dim = (hidden_dim * 2 * 2) if bidirectional else (hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(lstm_out_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, x):
        embeddings = self.lstm(x)
        return self.classifier(embeddings)

    
    def training_step(self, batch, batch_idx):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y, weight=self.class_weights, label_smoothing=0.1)
        self.train_acc(y_hat, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def validation_step(self, batch):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y, weight=self.class_weights, label_smoothing=0.1)
        self.val_acc(y_hat, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def test_step(self, batch):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        self.test_acc(y_hat, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)


    def predict_step(self, batch):
        x = batch['input_ids']
        return self(x)


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=5e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val-loss"
            }
        }

    
torch.manual_seed(42)  # Reproducibilidad: misma inicialización LSTM y clasificador
model = SpanishNewsClassifierWithLSTM(
    vocab_size=len(vocab),
    num_classes=wikicat_dataset.num_classes,
    emb_dim=EMB_DIM_SPACY,
    hidden_dim=128,
    num_layers=1,
    dropout=0.25,
    class_weights=class_weights_tensor,
    embedding_weights=spacy_embedding_matrix
)

tb_logger = TensorBoardLogger('tb_logs', name='LSTMClassifier_Optimized')
callbacks = [EarlyStopping(monitor='val-loss', patience=7, mode='min')]
trainer = Trainer(max_epochs=40, devices=1, logger=tb_logger, callbacks=callbacks, precision="16-mixed", gradient_clip_val=1.0)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4080 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm       │ LSTMBlock          │ 19.3 M │ train │     0 │
│ 1 │ classifier │ Sequential         │  921 K │ train │     0 │
│ 2 │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 4 │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 20.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 20.2 M                                                                                               
Total estimated model params size (MB): 80                                                                         
Modules in train mode: 18                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Observemos el proceso de entrenamiento

In [18]:
%load_ext tensorboard

In [19]:
%tensorboard --logdir tb_logs/

Reusing TensorBoard on port 6006 (pid 13128), started 13 days, 5:07:09 ago. (Use '!kill 13128' to kill it.)

Y como es de esperarse, realizaremos la validación contra el conjunto de prueba.

In [20]:
model.eval()
trainer.test(model, test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test-acc          │    0.7078986763954163     │
└───────────────────────────┴───────────────────────────┘

[{'test-acc': 0.7078986763954163}]

### Haciendo predicciones

Finalmente, vamos a hacer uso del modelo y ver qué tan bueno es para la clasificación temática de los artículos de Wikipedia del dataset WikiCAT_esv2.

In [21]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = predictions.numpy()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [22]:
import pandas as pd

# Índices originales dentro del dataset de HuggingFace
test_indices = test_dataset.indices
true_labels = dataset[test_indices]['label']

df = pd.DataFrame(data={
    "texto": dataset[test_indices]['text'],
    "tokens": [tokenize_text(v) for v in dataset[test_indices]['text']],
    "categoría": [wikicat_dataset.id2label[l] for l in true_labels],
    "predicción": [wikicat_dataset.id2label[p] for p in predictions],
}, index=test_indices)

df['tokens_string'] = df.tokens.apply(lambda t: ' '.join([id_2_token[i] for i in t]))
df = df[["texto", "tokens", "tokens_string", "categoría", "predicción"]]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

,texto,tokens,tokens_string,categoría,predicción
2247,"En Chile, los conservadores de bienes raíces (...","[4, 1243, 9, 2912, 2, 299, 1932, 38461, 26, 24...",en chile los conservadores de bienes raíces cb...,Derecho,Derecho
1472,Un proyecto de traducción es un proyecto que t...,"[10, 410, 2, 753, 13, 10, 410, 7, 276, 33, 3, ...",un proyecto de traducción es un proyecto que t...,Humanidades,Humanidades
4821,"Caudillo (del latín: capitellium, cabeza) es u...","[13589, 12, 352, 48410, 1907, 13, 10, 38, 1623...",caudillo del latín capitellium cabeza es un té...,Política,Política
3290,Como género musical tradicionalmente se conoce...,"[18, 201, 174, 1612, 11, 500, 18, 3090, 42287,...",como género musical tradicionalmente se conoce...,Música,Música
4785,Una política organizacional es un deliberado s...,"[15, 59, 6554, 13, 10, 13748, 58, 2, 249, 19, ...",una política organizacional es un deliberado s...,Política,Política
3450,"Gloria in excelsis Deo, llamado también doxolo...","[6774, 807, 26979, 14044, 337, 27, 43060, 114,...",gloria in excelsis deo llamado también doxolog...,Música,Música
6115,Las TÜV (Technischer Überwachungs-Verein) son ...,"[14, 11073, 1, 30166, 30167, 26, 390, 17190, 6...",las tüv [UNK] überwachungs verein son organiza...,Ciencia_y_Tecnología,Ciencia_y_Tecnología
2052,Redhibición es la acción por la que un comprad...,"[25533, 13, 3, 323, 16, 3, 7, 10, 3366, 47, 5,...",redhibición es la acción por la que un comprad...,Derecho,Derecho
2275,Un paralegal es el profesional de las ciencias...,"[10, 19697, 13, 5, 687, 2, 14, 136, 1787, 4, 8...",un paralegal es el profesional de las ciencias...,Derecho,Derecho
2852,El Centro de Investigación en Matemáticas A.C...,"[5, 331, 2, 108, 4, 199, 8, 147, 13965, 13, 10...",el centro de investigación en matemáticas a c ...,Matemáticas,Matemáticas


In [23]:
errors = df[df['categoría'] != df['predicción']]
errors.head(15)

,texto,tokens,tokens_string,categoría,predicción
5108,La expresión discusión bizantina o argumento b...,"[3, 268, 2399, 8903, 17, 1022, 10486, 315, 15,...",la expresión discusión bizantina o argumento b...,Religión,Historia
703,La investigación de mercados es la herramienta...,"[3, 108, 2, 1246, 13, 3, 1441, 1518, 19, 3, 15...",la investigación de mercados es la herramienta...,Economía,Ciencia_y_Tecnología
1381,Códigos 10 son palabras codificadas destinados...,"[2555, 26, 423, 15743, 4807, 8, 1710, 1379, 92...",códigos son palabras codificadas destinados a ...,Humanidades,Historia
2948,Las propiedades cualitativas son propiedades q...,"[14, 768, 10566, 26, 768, 7, 11, 10224, 6, 179...",las propiedades cualitativas son propiedades q...,Matemáticas,Economía
3257,"Sprezzatura es una palabra italiana, acuñada c...","[20365, 13, 15, 258, 2435, 5461, 18, 84, 2085,...",sprezzatura es una palabra italiana acuñada co...,Música,Política
3834,"La afinidad (taxonomía), principalmente en cie...","[3, 8224, 10388, 320, 4, 136, 2, 3, 135, 17, 4...",la afinidad taxonomía principalmente en cienci...,Filosofía,Humanidades
4938,El término legitimidad (y sus derivados: legít...,"[5, 38, 3010, 6, 30, 2839, 3612, 8, 11, 211, 4...",el término legitimidad y sus derivados legítim...,Política,Derecho
750,La observación de vida silvestre es la práctic...,"[3, 1559, 2, 135, 7046, 13, 3, 287, 2, 4777, 3...",la observación de vida silvestre es la práctic...,Entretenimiento,Ciencia_y_Tecnología
4580,X-Road es un software de código abierto que pe...,"[67, 4143, 13, 10, 906, 2, 639, 1444, 7, 291, ...",x road es un software de código abierto que pe...,Política,Ciencia_y_Tecnología
4762,La palabra veto procede del latín y significa ...,"[3, 258, 2083, 2067, 12, 352, 6, 315, 1506, 22...",la palabra veto procede del latín y significa ...,Política,Derecho


## Conclusiones

**Base (notebook original Sesión 2):** Clasificación multiclase en 12 categorías con LSTM bidireccional: vocabulario por frecuencia (50k tokens), embedding aleatorio, último estado de la LSTM (concatenación forward/backward) como representación del documento, clasificador denso 512→256→12. Arquitectura: 512 tokens de entrada, 2 capas LSTM, hidden 128.

**Mejoras añadidas en esta versión:**
- **Embeddings preentrenados (Spacy):** Vectores es_core_news_lg (300 dim), más vocabulario que _md. **Last hidden + mean pooling** sobre la secuencia (sin promediar padding) para usar todo el texto.
- **Pesos por clase:** Compensación del desbalance entre categorías en la pérdida.
- **Label smoothing (0.1):** Suavizado de las etiquetas para reducir sobreconfianza.
- **ReduceLROnPlateau:** Reducción del learning rate cuando la val-loss se estanca.
- **Early stopping** en val-loss (paciencia 7) y **batch_size=32** para un entrenamiento más estable.
- **Más capacidad:** secuencias **720 tokens**, LSTM 3 capas hidden 256, **last_hidden + mean pool** (1024 dim), clasificador 1024→512→512→256→12, LR 5e-4, gradient clipping 1.0, hasta **40 épocas** (patience 7), ReduceLROnPlateau patience 3.

**Resultado:** Con estas mejoras se puede alcanzar una **accuracy en test en torno al 75%** (o superior), por encima del baseline; destacan los vectores Spacy, la mayor capacidad del LSTM y las secuencias más largas.